# Filesystem, Network, SSRF, and Egress Security

Build two narrow MCP operations: read a catalogued file and look up a tenant ticket. The goal is not to sanitize arbitrary paths or URLs; it is to keep those authority-bearing values out of the tool contract.

This notebook performs real temporary-directory file reads. DNS and HTTP are deterministic offline fixtures, so no cloud credential or live network is used.

## Learning objectives

You will inspect opaque-ID MCP schemas, exercise descriptor-relative file opens, classify IPv4 and IPv6 destinations, prove connect-address pinning, attack redirect and response handling, and interpret redacted control evidence.

In [ ]:
import json
import runpy
from pathlib import Path
from mcp import Client

lab = runpy.run_path(Path('lab.py'))
environment = lab['default_environment']()
print('Loaded reusable lab components:', len(lab))

## 1. Start with the tool boundary

The caller selects a business resource, not a path, URL, host, tenant, or credential. Inspect the real MCP schemas before looking at implementation details.

In [ ]:
server = lab['build_mcp_server'](environment.service)
async with Client(server) as client:
    tools = await client.list_tools()
[(tool.name, tool.input_schema) for tool in tools.tools]

## 2. Read beneath an approved root

The file ID maps to a trusted tenant, root, relative path, digest, and size cap. The store opens each component relative to an already-open directory descriptor with no symlink following.

In [ ]:
valid_file = environment.file_store.read(environment.identity, 'file-support-policy')
valid_file

In [ ]:
file_results = {}
for file_id in ('file-other-tenant', 'file-symlink-escape', 'file-unknown'):
    try:
        environment.file_store.read(environment.identity, file_id)
        file_results[file_id] = 'unexpected allow'
    except lab['FileDenied'] as exc:
        file_results[file_id] = exc.code
file_results

A digest validates the bytes actually opened, but it does not replace containment. Production Linux deployments should consider `openat2` resolution flags and still mount only necessary input read-only inside an isolated runtime.

## 3. Classify resolved addresses

The policy validates every A and AAAA answer. IPv4-mapped IPv6 is normalized before classification.

In [ ]:
addresses = ['93.184.216.34', '127.0.0.1', '10.0.0.1', '169.254.169.254', '::1', 'fe80::1', '::ffff:169.254.169.254']
{value: lab['address_class'](value) for value in addresses}

In [ ]:
approved = environment.gate.approve(
    lab['canonical_ticket_url']('acme-100'),
    environment.destination,
    request_id='net-notebook-approve',
    hop=0,
)
{'connect_ip': approved.connect_ip, 'tls_server_name': approved.host, 'path': approved.path}

In [ ]:
metadata_resolver = lab['ScriptedResolver']({lab['TICKET_HOST']: ((lab['METADATA_IP'],),)})
metadata_gate = lab['URLGate'](metadata_resolver)
try:
    metadata_gate.approve(lab['canonical_ticket_url']('acme-100'), environment.destination, request_id='net-metadata', hop=0)
    metadata_result = 'unexpected allow'
except lab['EgressDenied'] as exc:
    metadata_result = exc.code
metadata_result

## 4. Re-resolve and re-authorize every redirect

A public answer on the first hop does not authorize a private answer on the next. The transport receives the vetted IP and retains the logical hostname for TLS verification.

In [ ]:
resolver = lab['ScriptedResolver']({lab['TICKET_HOST']: ((lab['PUBLIC_TICKET_IP'],), (lab['METADATA_IP'],))})
gate = lab['URLGate'](resolver)
transport = lab['ScriptedTransport']()
start_url = lab['canonical_ticket_url']('acme-start')
transport.add(connect_ip=lab['PUBLIC_TICKET_IP'], method='GET', url=start_url, result=lab['FakeResponse'](302, {'Location': '/v1/tickets/acme-next'}))
client = lab['EgressClient']((environment.destination,), gate, transport, lab['CredentialBroker']({'tickets-api': 'notebook-secret'}))
try:
    client.fetch_json(lab['EgressRequest'](request_id='net-rebind', destination_id='dest-tickets-api', path='/v1/tickets/acme-start'))
    rebinding_result = 'unexpected allow'
except lab['EgressDenied'] as exc:
    rebinding_result = exc.code
{'result': rebinding_result, 'transport_calls': len(transport.calls), 'dns_lookups': resolver.calls}

## 5. Bound and bind the response

The destination may still fail or be compromised. Status, media type, declared and streamed size, JSON shape, ticket ID, and tenant are independently checked.

In [ ]:
ticket = environment.service.lookup_ticket('acme-100')
ticket

In [ ]:
fault_env = lab['default_environment']()
fault_env.transport.routes.clear()
fault_env.transport.add(
    connect_ip=lab['PUBLIC_TICKET_IP'], method='GET',
    url=lab['canonical_ticket_url']('acme-100'),
    result=lab['FakeResponse'](200, {'Content-Type': 'application/json'}, (b'x' * 5000,)),
)
try:
    fault_env.service.lookup_ticket('acme-100')
    response_result = 'unexpected allow'
except lab['EgressDenied'] as exc:
    response_result = exc.code
finally:
    fault_env.close()
response_result

## 6. Inspect redacted evidence

Evidence names the logical destination, decision, address class, hop, size, and credential fingerprint. It omits file contents, raw query values, and bearer tokens.

In [ ]:
events = lab['serialized_events'](environment)
assert environment.raw_token not in events
json.loads(events)[-4:]

## 7. Run the full MCP scenario

This uses the real in-memory MCP protocol path and asserts zero completed forbidden effects.

In [ ]:
evidence = await lab['run_scenario']()
evidence

In [ ]:
environment.close()
assert evidence['forbidden_effects_observed'] == 0
print('Temporary fixture removed; all scenario invariants held.')

## Reflection

1. Why is a validated hostname insufficient unless the connect address is pinned?
2. What authority remains in an opaque ticket ID, and where is it checked?
3. Why must every DNS answer pass rather than one answer?
4. Which redirect properties require fresh validation?
5. Which lab claims are simulations, and what deployment evidence would prove the production network boundary?